In [2]:
import numpy as np
import pandas as pd
import time

class display(object):
    """Display HTML representation of multiple objects"""
    template = """<div style="float: left; padding: 10px;">
    <p style='font-family:"Courier New", Courier, monospace'>{0}</p>{1}
    </div>"""
    def __init__(self, *args):
        self.args = args
        
    def _repr_html_(self):
        rendered = []
        for a in self.args:
            try:
                rendered.append(self.template.format(a, eval(a)._repr_html_()))
            except Exception as e:
                rendered.append(self.template.format(a, f"<pre>Error: {e}</pre>"))
        return '\n'.join(rendered)
    
    def __repr__(self):
        lines = []
        for a in self.args:
            try:
                lines.append(a + '\n' + repr(eval(a)))
            except Exception as e:
                lines.append(f"{a}\nError: {e}")
        return '\n\n'.join(lines)

# pandas.eval for efficient operations

In [3]:
df1 = pd.DataFrame(np.random.randn(1_000_000, 10))
df2 = pd.DataFrame(np.random.randn(1_000_000, 10))
start = time.time()
pd.eval('df1 / 2.3 + 5 * df2')
end = time.time()
print(f"Time taken for pd.eval: {(end - start) * 1000} milliseconds")
start = time.time()
df1 / 2.3 + 5 * df2
end = time.time()
print(f"Time taken for standard addition: {(end - start) * 1000} milliseconds")

Time taken for pd.eval: 113.63387107849121 milliseconds
Time taken for standard addition: 108.56318473815918 milliseconds


In [11]:
df1 = pd.DataFrame(np.random.randn(100_000, 5))
df2 = pd.DataFrame(np.random.randn(100_000, 5))
pd.eval('df1 / 2.3 + 5 * df2').head()

,0,1,2,3,4
0,4.213646,-3.622166,-1.075384,-6.978271,0.396647
1,8.369166,3.846971,-1.026112,2.423708,-7.836343
2,-1.831083,-8.504885,-2.592960,-4.819442,5.757949
3,-2.593972,-7.231471,-11.647101,2.055810,8.101563
4,9.855727,3.919002,2.580563,0.929714,-3.721500


## operators supported by pandas.eval
- Arithmetic operators: `+`, `-`, `*`, `/`, `**`, `%`, `//`
- Comparison operators: `==`, `!=`, `<`, `>`, `<=`, `>=`
- Bitwise operators: `&`, `|`
- object attributes: `df.column_name` or `df['column_name']`

In [4]:
df1, df2, df3, df4 = [pd.DataFrame(np.random.randn(5, 5), columns=list('ABCDE')) for _ in range(4)]

print(pd.eval('df1 + df2 - df3 * df4'))
print(pd.eval('(df1 >= df3) | (df2 <= df4)'))
print(pd.eval('df1.T'))

          A         B         C         D         E
0  2.081671 -0.448362  0.366912  0.893038  0.774659
1  0.065653 -0.471833 -0.751465 -0.578824  1.535102
2 -5.013821 -3.353259  0.872956  0.014244  1.304423
3  3.554100  0.416243  2.915022  0.367519  0.821830
4  1.032441  1.408209  0.073609  2.253118 -0.807038
       A      B      C      D      E
0   True  False   True   True   True
1   True   True  False  False   True
2  False   True  False  False   True
3   True   True   True   True  False
4  False  False   True   True  False
          0         1         2         3         4
A  1.009403 -0.931938 -2.479118  2.211075 -1.434363
B -0.406361 -0.357066  0.450608  1.151103 -0.243251
C  0.139580 -0.317731 -0.530953  1.380668  0.812106
D  0.210915 -2.797556  0.030871 -0.449825  0.080073
E  0.705854  2.708708  1.187392 -0.414723 -1.114293


# dataframe.eval for column-wise operations

In [16]:
df = pd.DataFrame(np.random.randint(10, size=(10, 3)), columns=list('ABC'))
display('df', "df.eval('A + B * C').to_frame()")

,A,B,C
0,9,9,5
1,9,1,6
2,9,3,2
3,4,1,4
4,7,2,7
5,5,6,3
6,3,3,8
7,7,2,4
8,6,0,1
9,7,6,8


## assignment in dataframe.eval

In [6]:
df = pd.DataFrame(np.random.randn(5, 3), columns=list('ABE'))
display('df', "df.eval('F = 99 * A + B * E')")

df
          A         B         E
0  0.694658  0.306501  0.986995
1 -1.516570  0.188445  0.529333
2  0.038076 -1.245260  0.199539
3 -0.171271  1.144590 -1.017873
4  0.221082 -1.786084  0.857043

df.eval('F = 99 * A + B * E')
          A         B         E           F
0  0.694658  0.306501  0.986995   69.073706
1 -1.516570  0.188445  0.529333 -150.040664
2  0.038076 -1.245260  0.199539    3.521002
3 -0.171271  1.144590 -1.017873  -18.120881
4  0.221082 -1.786084  0.857043   20.356350

## local variables in dataframe.eval

In [7]:
num = 0x99999
display('df', "df.eval('F = A + @num')")

df
          A         B         E
0  0.694658  0.306501  0.986995
1 -1.516570  0.188445  0.529333
2  0.038076 -1.245260  0.199539
3 -0.171271  1.144590 -1.017873
4  0.221082 -1.786084  0.857043

df.eval('F = A + @num')
          A         B         E              F
0  0.694658  0.306501  0.986995  629145.694658
1 -1.516570  0.188445  0.529333  629143.483430
2  0.038076 -1.245260  0.199539  629145.038076
3 -0.171271  1.144590 -1.017873  629144.828729
4  0.221082 -1.786084  0.857043  629145.221082

# dataframe.query and instance query function

In [8]:
display('df', 'df.query("A > 0 and B < 0")', 'pd.eval("df.query(\'A > 0 and B < 0\')")')

,A,B,E
0,0.694658,0.306501,0.986995
1,-1.516570,0.188445,0.529333
2,0.038076,-1.245260,0.199539
3,-0.171271,1.144590,-1.017873
4,0.221082,-1.786084,0.857043
,A,B,E
2,0.038076,-1.245260,0.199539
4,0.221082,-1.786084,0.857043
,A,B,E
2,0.038076,-1.245260,0.199539


# performance: when to use these functions

if the data is too large, use the eval and query. the overhead in time and storage of new dataframes for traditional operations can add up. The eval and query functions are optimized for performance and can handle large datasets more efficiently. for small datasets, the performance is better in traditional operations.